# 04 — Self-Balancing Trees and Tries

## Why This Matters for DSA
A standard Binary Search Tree (`02_Trees_and_BST`) guarantees $O(\log n)$ operations *only* if the tree remains reasonably balanced. If elements are inserted in sorted order, the BST degenerates into a linear chain, destroying search performance ($O(n)$ worst-case). **Self-Balancing BSTs**, specifically AVL Trees, solve this degeneracy problem by automatically restructuring themselves during insertions and deletions, guaranteeing $O(\log n)$ performance in the strict mathematical worst-case. In parallel, string search workloads benefit from a specialized structure: the **Trie (Prefix Tree)**. While trees and hash maps depend on key comparison or hash calculations, Tries organize strings by their prefixes, allowing lookup times that depend solely on the search string's length ($O(L)$), making them the industry standard for search dictionaries and autocomplete engines.

## Prerequisites
- `07_Linked_Lists` (Phase 02) — dynamic pointer manipulation
- `02_Trees_and_BST` (Phase 03) — BST invariants, node deletion logic, and recursive operations
- `03_Tree_DFS_BFS` (Phase 03) — DFS backtracking traversal pattern

## Learning Objectives
By the end of this notebook, you will be able to:
- State the balance invariant of an **AVL Tree** and calculate node balance factors ($BF = h_{left} - h_{right}$).
- Identify the four imbalance cases (**LL, RR, LR, RL**) and apply the corresponding single or double rotations to restore balance.
- Explain the critical differences between AVL insertion (at most one rotation) and deletion (potentially $O(h)$ rotations) in balancing behavior.
- Implement a complete AVL Tree in C++ with recursive insertion, height updates, and tree rotations.
- Design and implement a **Trie (Prefix Tree)** with `insert`, `search`, and `startsWith` operations.
- Compare BSTs, Hash Maps, AVL Trees, and Tries on time-space complexity for string keys.
- Build a Trie-based autocomplete prefix matching system using DFS.

## Checklist Before Moving On
- [ ] I can calculate the balance factor of any tree node by hand
- [ ] I can trace the four rotation cases (LL, RR, LR, RL) on paper
- [ ] I can explain why AVL deletion can trigger multiple rotations up to the root while insertion triggers at most one
- [ ] I can write the C++ rotation equations for `SingleRotateWithLeft` and `SingleRotateWithRight` from memory
- [ ] I can define the Trie node structure and write its child pointer representation
- [ ] I can write the recursive insert and search functions for a Trie
- [ ] I can explain why Trie lookups are $O(L)$ (word length) independent of total stored items

## Table of Contents
1. The Need for Balance — AVL Definition and Balance Factors
2. AVL Rotations — LL, RR, LR, and RL Transformations
3. AVL Insertion and Deletion Mechanics
4. Complete C++ AVL Tree Implementation
5. Trie (Prefix Tree) Fundamentals
6. Complete C++ Trie Implementation
7. String Operations Comparison Matrix — BST vs. Hash Map vs. AVL vs. Trie
8. Decision Guide — AVL vs. Red-Black vs. Trie
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems


## 1. The Need for Balance — AVL Definition and Balance Factors

### The Why
Standard BSTs (`02_Trees_and_BST` Section 7) can easily degenerate into skewed linear chains if keys are inserted in sorted order. If a BST becomes unbalanced, search, insertion, and deletion degrade from $O(\log n)$ to $O(n)$ time. To prevent this, Adelson-Velskii and Landis designed the **AVL Tree**—a self-balancing BST that maintains a height-balance invariant at every node, ensuring logarithmic heights in all cases.

### Core Mechanism
- **The Balance Invariant:**
  A binary search tree is AVL-balanced if:
  1. The heights of its left and right subtrees differ by **at most 1**.
  2. Both subtrees are themselves AVL trees.
- **Balance Factor (BF):**
  The balance factor of a node is the height of its left subtree minus the height of its right subtree:
  $$BF(N) = Height(N.left) - Height(N.right)$$
  - In an AVL Tree, the balance factor of *every* node must be in the set **$\{-1, 0, 1\}$**.
  - If $BF(N) = 2$ or $BF(N) = -2$ at any node, the tree is out of balance and a rotation transformation must be performed.
- **Height Definitions:**
  - An empty tree (null pointer) has height **-1**.
  - A single node tree has height **0**.
  - For any node $N$: $Height(N) = 1 + \max(Height(N.left), Height(N.right))$.

### Common Pitfalls
- **Confusing height and balance factor:** Height is the maximum distance from a node to a leaf node. Balance factor is the difference between subtree heights ($h_{left} - h_{right}$). Balance factor is calculated using height, not node counts.
- **Forgetting the empty tree base case:** Assuming empty subtrees have height 0. Empty trees must be defined with height -1; otherwise, a single node with no children would have $BF = 0 - 0 = 0$ but $Height = 1 + 0 = 1$ (which violates the height 0 standard).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <algorithm>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left, *right;
    int height;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr), height(0) {}
};

int getHeight(TreeNode* node) {
    return node == nullptr ? -1 : node->height;
}

int getBalanceFactor(TreeNode* node) {
    if (node == nullptr) return 0;
    return getHeight(node->left) - getHeight(node->right);
}

int main() {
    // Construct an unbalanced BST:
    //       30 (height 2, BF = 2 - 0 = 2) <- Unbalanced!
    //      / 
    //     20  (height 1, BF = 1 - (-1) = 2) <- Unbalanced!
    //    /   
    //   10   (height 0, BF = 0)
    TreeNode* root = new TreeNode(30);
    root->left = new TreeNode(20);
    root->left->left = new TreeNode(10);
    
    // Update heights manually for demonstration
    root->left->left->height = 0;
    root->left->height = 1 + max(getHeight(root->left->left), getHeight(root->left->right));
    root->height = 1 + max(getHeight(root->left), getHeight(root->right));

    cout << "=== Unbalanced Tree Check ===\n";
    cout << "Node 10 Height: " << getHeight(root->left->left) << " BF: " << getBalanceFactor(root->left->left) << "\n"; // Height 0, BF 0
    cout << "Node 20 Height: " << getHeight(root->left) << " BF: " << getBalanceFactor(root->left) << "\n";       // Height 1, BF 2
    cout << "Node 30 Height: " << getHeight(root) << " BF: " << getBalanceFactor(root) << "\n";             // Height 2, BF 2
    
    // Clean up
    delete root->left->left;
    delete root->left;
    delete root;
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. AVL Rotations — LL, RR, LR, and RL Transformations

### The Why
When an insertion or deletion breaks the AVL balance invariant (causing a balance factor of $2$ or $-2$ at node $A$), we must restructure the tree locally. We do this using **rotations**—relinking pointers so that the height is reduced while preserving the binary search inorder relationship. Imbalances fall into four cases depending on which subtrees caused the height increase.

### Core Mechanism
We classify imbalances into four cases relative to the first unbalanced node $A$:
1. **Left-Left (LL) Case:** Insertion into the left subtree of the left child of $A$. The left subtree is too heavy ($BF(A) = 2$).
   - **Solution: Single Right Rotation** around $A$ (in slide code, `SingleRotateWithLeft`).
2. **Right-Right (RR) Case:** Insertion into the right subtree of the right child of $A$. The right subtree is too heavy ($BF(A) = -2$).
   - **Solution: Single Left Rotation** around $A$ (in slide code, `SingleRotateWithRight`).
3. **Left-Right (LR) Case:** Insertion into the right subtree of the left child of $A$. The left subtree is too heavy, and its right child is deeper.
   - **Solution: Double Left-Right Rotation** (in slide code, `DoubleRotateWithLeft`). Perform a single left rotation around $A.left$, then a single right rotation around $A$.
4. **Right-Left (RL) Case:** Insertion into the left subtree of the right child of $A$. The right subtree is too heavy, and its left child is deeper.
   - **Solution: Double Right-Left Rotation** (in slide code, `DoubleRotateWithRight`). Perform a single right rotation around $A.right$, then a single left rotation around $A$.

```mermaid
graph TD
    subgraph "Single Right Rotation (LL Case)"
    A((A)) --> B((B))
    A --> AR[AR]
    B --> C((C))
    B --> BR[BR]
    style A fill:#f9f,stroke:#333
    end
    
    subgraph "Rotates Right To"
    B2((B)) --> C2((C))
    B2 --> A2((A))
    A2 --> BR2[BR]
    A2 --> AR2[AR]
    end
```

### Common Pitfalls
- **Misidentifying the Rotation Case:** Trying to fix an LR imbalance with a single right rotation. If you do this, the tree will remain unbalanced ($BF$ just shifts to the other side). Always inspect the balance factor of the unbalanced node *and* the balance factor of its heavy child to decide between single and double rotations.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <algorithm>
using namespace std;

struct AvlNode {
    int Element;
    AvlNode *Left, *Right;
    int Height;
    AvlNode(int val) : Element(val), Left(nullptr), Right(nullptr), Height(0) {}
};

int Height(AvlNode* n) {
    return n == nullptr ? -1 : n->Height;
}

// SINGLE ROTATION LEFT (RR Case): Rotates right child K2 up, K1 becomes left child
// Slide: "SingleRotateWithRight"
AvlNode* SingleRotateWithRight(AvlNode* K1) {
    AvlNode* K2 = K1->Right;
    K1->Right = K2->Left;
    K2->Left = K1;
    
    K1->Height = max(Height(K1->Left), Height(K1->Right)) + 1;
    K2->Height = max(Height(K2->Right), K1->Height) + 1;
    return K2; // Return new root of subtree
}

// SINGLE ROTATION RIGHT (LL Case): Rotates left child K1 up, K2 becomes right child
// Slide: "SingleRotateWithLeft"
AvlNode* SingleRotateWithLeft(AvlNode* K2) {
    AvlNode* K1 = K2->Left;
    K2->Left = K1->Right;
    K1->Right = K2;
    
    K2->Height = max(Height(K2->Left), Height(K2->Right)) + 1;
    K1->Height = max(Height(K1->Left), K2->Height) + 1;
    return K1; // Return new root of subtree
}

// DOUBLE ROTATION LEFT-RIGHT (LR Case): Rotate left child left, then rotate root right
// Slide: "DoubleRotateWithLeft"
AvlNode* DoubleRotateWithLeft(AvlNode* K3) {
    K3->Left = SingleRotateWithRight(K3->Left);
    return SingleRotateWithLeft(K3);
}

// DOUBLE ROTATION RIGHT-LEFT (RL Case): Rotate right child right, then rotate root left
// Slide: "DoubleRotateWithRight"
AvlNode* DoubleRotateWithRight(AvlNode* K1) {
    K1->Right = SingleRotateWithLeft(K1->Right);
    return SingleRotateWithRight(K1);
}

void printPreOrder(AvlNode* node) {
    if (node) {
        cout << node->Element << " ";
        printPreOrder(node->Left);
        printPreOrder(node->Right);
    }
}

int main() {
    // Test right rotation (LL Case): 
    //       30
    //      /   
    //     20
    //    /  
    //   10
    AvlNode* root = new AvlNode(30);
    root->Left = new AvlNode(20);
    root->Left->Left = new AvlNode(10);
    root->Left->Left->Height = 0;
    root->Left->Height = 1;
    root->Height = 2;

    cout << "Original LL path preorder: ";
    printPreOrder(root); cout << "\n"; // 30 20 10
    
    root = SingleRotateWithLeft(root);
    cout << "After Right rotation:      ";
    printPreOrder(root); cout << "\n"; // 20 10 30 (Balanced!)

    delete root->Left;
    delete root->Right;
    delete root;
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. AVL Insertion and Deletion Mechanics

### The Why
While AVL insertion and deletion are modeled on standard BST operations, they differ in how they restore balance. When you insert a node, a rotation at the lowest unbalanced ancestor restores the balance of that subtree *and* returns it to its pre-insertion height. This means insertion requires **at most one** single/double rotation. In contrast, deleting a node can shorten a subtree, which can propagate imbalances up the tree. Consequently, deletion may require **up to $O(h)$ rotations** along the path to the root.

### Core Mechanism
- **AVL Insertion:**
  1. Perform standard recursive BST insertion.
  2. On the way back up (unwinding the recursion stack), update the node height.
  3. Check the balance factor ($BF$). If it is $2$ or $-2$:
     - Determine the rotation case (LL, RR, LR, RL) based on where the value was inserted.
     - Perform the rotation and return the new root of this subtree.
     - Once a rotation is performed, the subtree height matches its height before the insertion, meaning ancestors do not need rebalancing. Insertion triggers at most one rotation.
- **AVL Deletion:**
  1. Perform standard recursive BST deletion.
  2. On the way back up, update the node height.
  3. Check the balance factor. If it is $2$ or $-2$, perform the appropriate rotation.
  4. **The Difference:** Even after rotating, the subtree height may remain shorter than before the deletion. This can create an imbalance at the grandparent node, then the great-grandparent, and so on. We must check and rotate at *every* ancestor on the path back to the root, taking up to $O(\log n)$ rotations.

### Common Pitfalls
- **Assuming Deletion is Single-Rotation:** Believing that a single rotation is sufficient to balance the tree after deletion. If you stop rebalancing after the first rotation during deletion, the tree will remain unbalanced further up, violating the AVL invariant. Always loop/recurse up to the root.


## 4. Complete C++ AVL Tree Implementation

### The Why
Seeing a complete C++ class implementation is critical for understanding how rotations, height calculations, and recursion link together. This class demonstrates an insert-only AVL Tree, showcasing the complete workflow from node creation to recursive balancing.

### Core Mechanism
- The `Insert` function is recursive, returning the updated node pointer (`AvlTree`) up the call stack to reconstruct child links automatically, mimicking standard C++ styles.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <algorithm>
using namespace std;

struct AvlNode {
    int Element;
    AvlNode* Left;
    AvlNode* Right;
    int Height;
    AvlNode(int x) : Element(x), Left(nullptr), Right(nullptr), Height(0) {}
};

typedef AvlNode* Position;
typedef AvlNode* AvlTree;

int Height(Position P) {
    return P == nullptr ? -1 : P->Height;
}

Position SingleRotateWithLeft(Position K2) {
    Position K1 = K2->Left;
    K2->Left = K1->Right;
    K1->Right = K2;
    
    K2->Height = max(Height(K2->Left), Height(K2->Right)) + 1;
    K1->Height = max(Height(K1->Left), K2->Height) + 1;
    return K1;
}

Position SingleRotateWithRight(Position K1) {
    Position K2 = K1->Right;
    K1->Right = K2->Left;
    K2->Left = K1;
    
    K1->Height = max(Height(K1->Left), Height(K1->Right)) + 1;
    K2->Height = max(Height(K2->Right), K1->Height) + 1;
    return K2;
}

Position DoubleRotateWithLeft(Position K3) {
    K3->Left = SingleRotateWithRight(K3->Left);
    return SingleRotateWithLeft(K3);
}

Position DoubleRotateWithRight(Position K1) {
    K1->Right = SingleRotateWithLeft(K1->Right);
    return SingleRotateWithRight(K1);
}

AvlTree Insert(int X, AvlTree T) {
    if (T == nullptr) {
        T = new AvlNode(X);
    }
    else if (X < T->Element) {
        T->Left = Insert(X, T->Left);
        if (Height(T->Left) - Height(T->Right) == 2) {
            if (X < T->Left->Element)
                T = SingleRotateWithLeft(T);
            else
                T = DoubleRotateWithLeft(T);
        }
    }
    else if (X > T->Element) {
        T->Right = Insert(X, T->Right);
        if (Height(T->Right) - Height(T->Left) == 2) {
            if (X > T->Right->Element)
                T = SingleRotateWithRight(T);
            else
                T = DoubleRotateWithRight(T);
        }
    }
    
    T->Height = max(Height(T->Left), Height(T->Right)) + 1;
    return T;
}

void printInOrder(AvlTree T) {
    if (T != nullptr) {
        printInOrder(T->Left);
        cout << T->Element << " ";
        printInOrder(T->Right);
    }
}

void destroyTree(AvlTree T) {
    if (T != nullptr) {
        destroyTree(T->Left);
        destroyTree(T->Right);
        delete T;
    }
}

int main() {
    AvlTree t = nullptr;
    
    // Build AVL tree using complete set from slides:
    // 3, 2, 1, 4, 5, 6, 7
    vector<int> elements = {3, 2, 1, 4, 5, 6, 7};
    for (int x : elements) {
        t = Insert(x, t);
    }

    cout << "AVL Tree Inorder traversal: ";
    printInOrder(t); cout << "\n"; // Expected: 1 2 3 4 5 6 7
    cout << "Root node element:          " << t->Element << "\n"; // Expected: 4
    cout << "Root height (logarithmic):  " << Height(t) << "\n";    // Expected: 2

    destroyTree(t);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Trie (Prefix Tree) Fundamentals

### The Why
Binary search trees and hash maps lookup keys by comparing values or calculating hashes. For string keys, this means lookup times are proportional to string length ($O(L)$) *plus* potential collisions or tree traversal depth ($O(L \log n)$). A **Trie (Prefix Tree)** is a specialized retrieval structure where characters of strings are mapped to nodes. This allows word lookups and prefix match queries to execute in $O(L)$ time, completely independent of the number of words stored in the tree.

### Core Mechanism
- **Structure:**
  - A Trie is a multi-way tree. Each node contains:
    1. An array of child pointers (typically of size 26 for lowercase English characters, mapping indices $0..25$ to characters $'a'..'z'$).
    2. A boolean flag `isEndOfWord`, which is set to true at nodes that represent the end of a valid word.
  - Paths from the root represent prefixes. Words with shared prefixes share the same node path. For example, "app" and "apple" share the path for 'a' -> 'p' -> 'p'.
- **Operations:**
  - `insert(word)`: Start at the root. For each character, move to the child node corresponding to that character, allocating new nodes if they do not exist. Mark the final character's node as `isEndOfWord = true`.
  - `search(word)`: Traverse the character path. If a child pointer is null, return `false`. At the final character, return `isEndOfWord`.
  - `startsWith(prefix)`: Traverse the prefix path. If the path is successfully traversed (no null pointers are encountered), return `true`.

### Common Pitfalls
- **Wasted Memory:** Using a raw child array of size 26 for every node. If the vocabulary is sparse, most pointers will be null, consuming substantial memory. For memory optimization, a hash map (`std::unordered_map<char, TrieNode*>`) or sorted vector can replace the static array.
- **Forgetting the End of Word Flag:** Checking if a path exists instead of verifying the `isEndOfWord` flag during a search. If you search for "ap" in a Trie containing only "apple", the path exists, but it is not a valid word, so it must return `false`.


## 6. Complete C++ Trie Implementation

### The Why
A clean C++ implementation of a Trie shows how character arithmetic (`c - 'a'`) is used to map characters to child indices, and how we traverse nodes iteratively.

### Core Mechanism
- Standard iterative code for Trie operations, featuring memory cleanup in the destructor.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
using namespace std;

struct TrieNode {
    TrieNode* children[26];
    bool isEndOfWord;
    TrieNode() {
        for (int i = 0; i < 26; i++) children[i] = nullptr;
        isEndOfWord = false;
    }
};

class Trie {
private:
    TrieNode* root;

    void clear(TrieNode* node) {
        if (node == nullptr) return;
        for (int i = 0; i < 26; i++) {
            clear(node->children[i]);
        }
        delete node;
    }

public:
    Trie() { root = new TrieNode(); }
    ~Trie() { clear(root); }

    // Insert a word: O(L) time, O(L) space (worst-case)
    void insert(string word) {
        TrieNode* cur = root;
        for (char c : word) {
            int idx = c - 'a'; // Map 'a'..'z' to 0..25
            if (cur->children[idx] == nullptr) {
                cur->children[idx] = new TrieNode();
            }
            cur = cur->children[idx];
        }
        cur->isEndOfWord = true;
    }

    // Search for a full word: O(L) time, O(1) auxiliary space
    bool search(string word) {
        TrieNode* cur = root;
        for (char c : word) {
            int idx = c - 'a';
            if (cur->children[idx] == nullptr) return false;
            cur = cur->children[idx];
        }
        return cur->isEndOfWord;
    }

    // Check prefix existence: O(L) time, O(1) auxiliary space
    bool startsWith(string prefix) {
        TrieNode* cur = root;
        for (char c : prefix) {
            int idx = c - 'a';
            if (cur->children[idx] == nullptr) return false;
            cur = cur->children[idx];
        }
        return true;
    }
};

int main() {
    Trie t;
    t.insert("apple");
    t.insert("app");

    cout << "Search 'apple': " << t.search("apple") << "\n";   // Expected: 1 (true)
    cout << "Search 'app':   " << t.search("app") << "\n";     // Expected: 1 (true)
    cout << "Search 'ap':    " << t.search("ap") << "\n";      // Expected: 0 (false)
    cout << "StartsWith 'ap':" << t.startsWith("ap") << "\n";  // Expected: 1 (true)

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. String Operations Complexity Comparison Matrix

### The Why
Evaluating when to use a Trie over general BSTs or Hash Maps for string key management is a core design skill. This matrix evaluates their runtime characteristics, highlighting the trade-offs.

### Complexity Matrix

For $n$ keys of average length $L$:

| Data Structure | Search / Lookup | Insertion | Space Complexity | Prefix Matching (`startsWith`) |
|---|---|---|---|---|
| **Balanced BST** | $O(L \log n)$ | $O(L \log n)$ | $O(n L)$ | $O(L \log n + k)$ (via range search) |
| **Hash Map** | $O(L)$ (average) | $O(L)$ (average) | $O(n L)$ | $O(n L)$ (must scan all keys) |
| **AVL Tree** | $O(L \log n)$ | $O(L \log n)$ | $O(n L)$ | $O(L \log n + k)$ |
| **Trie** | $O(L)$ (worst-case) | $O(L)$ (worst-case) | $O(n \Sigma L)$ (high overhead) | $O(L)$ (optimal) |

*Note: $\Sigma$ represents the alphabet size (e.g., 26 for lowercase English). Hash map search requires hashing the string, which takes $O(L)$ time; thus, Trie lookup is comparable to hash maps but offers sorted prefix queries, which hash maps cannot do.*


## 8. Decision Guide — AVL vs. Red-Black vs. Trie

### The Why
Different tree structures optimize different workloads. This guide outlines the engineering trade-offs.

### Decision Rules
- **Choose AVL Trees when:**
  - Your workload is dominated by lookups/searches rather than updates. AVL Trees are more strictly balanced than Red-Black Trees, resulting in shorter heights and faster average search times.
- **Choose Red-Black Trees when:**
  - Your workload is write-heavy (frequent insertions/deletions). Red-Black Trees have looser balance constraints, meaning they trigger fewer rotations during insertions/deletions than AVL Trees. (Note: standard templates like C++ `std::map` are backed by Red-Black Trees).
- **Choose Tries when:**
  - Your keys are strings, and you need to perform prefix-based lookups (e.g., autocomplete suggestions, spell checkers, prefix IP routing).
  - You want lookup times to remain independent of the size of the dataset.


## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### Checkpoint Questions
1. Define the balance factor of an AVL Tree node, and list the valid balance factors.
2. Identify the rotation case and action needed when node $A$ has balance factor $2$ and its left child has balance factor $-1$.
3. Explain why AVL deletion can require up to $O(h)$ rotations while insertion requires at most one.
4. Why does a Trie search for word $W$ execute in $O(L)$ time regardless of whether there are $100$ or $10^6$ words stored in the structure?
5. When is a Trie less space-efficient than a standard Hash Map?

### Answer Key
1. Balance factor $BF(N) = Height(N.left) - Height(N.right)$. Valid balance factors are $-1$, $0$, and $1$.
2. This is the Left-Right (LR) case. It requires a Double Left-Right Rotation: first a single left rotation around the left child, then a single right rotation around node $A$.
3. Inserting a node increases height. Rebalancing that subtree restores its height to the pre-insertion height, resolving any imbalances further up. Deleting a node decreases height. Rebalancing the subtree can shorten it, propagating imbalances to grandparent and higher ancestor nodes, requiring up to $O(h)$ rotations.
4. Trie search traverses a single path matching the characters of $W$. We do not perform any key comparisons against other elements; we simply inspect the child pointer array at each character step, taking exactly $L$ steps.
5. When words are highly unique and do not share common prefixes. Each node in the Trie allocates a full array of child pointers, leading to massive memory overhead compared to a flat hash map.

### Mini Project: Trie Autocomplete Suggestions
Implement an autocomplete system: given a query prefix, return all words stored in the Trie that begin with that prefix. This combines prefix traversal with DFS backtracking (`03_Tree_DFS_BFS` Section 2) to collect matches.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
#include <vector>
using namespace std;

struct TrieNode {
    TrieNode* children[26];
    bool isEndOfWord;
    TrieNode() {
        for (int i = 0; i < 26; i++) children[i] = nullptr;
        isEndOfWord = false;
    }
};

class AutocompleteTrie {
private:
    TrieNode* root;

    // DFS to collect all words starting from a given node
    void collectWords(TrieNode* node, string currentPrefix, vector<string>& results) {
        if (node == nullptr) return;
        if (node->isEndOfWord) {
            results.push_back(currentPrefix);
        }
        for (int i = 0; i < 26; i++) {
            if (node->children[i] != nullptr) {
                char nextChar = 'a' + i;
                collectWords(node->children[i], currentPrefix + nextChar, results);
            }
        }
    }

    void clear(TrieNode* node) {
        if (node == nullptr) return;
        for (int i = 0; i < 26; i++) clear(node->children[i]);
        delete node;
    }

public:
    AutocompleteTrie() { root = new TrieNode(); }
    ~AutocompleteTrie() { clear(root); }

    void insert(string word) {
        TrieNode* cur = root;
        for (char c : word) {
            int idx = c - 'a';
            if (cur->children[idx] == nullptr) cur->children[idx] = new TrieNode();
            cur = cur->children[idx];
        }
        cur->isEndOfWord = true;
    }

    // Autocomplete query: returns all words with given prefix
    vector<string> getSuggestions(string prefix) {
        vector<string> results;
        TrieNode* cur = root;
        for (char c : prefix) {
            int idx = c - 'a';
            if (cur->children[idx] == nullptr) return results; // No words match prefix
            cur = cur->children[idx];
        }
        
        // Start DFS from prefix node to find all matching suffix words
        collectWords(cur, prefix, results);
        return results;
    }
};

int main() {
    AutocompleteTrie trie;
    trie.insert("car");
    trie.insert("cart");
    trie.insert("care");
    trie.insert("dog");
    trie.insert("cat");

    vector<string> suggestions = trie.getSuggestions("car");
    cout << "Suggestions for 'car':\n";
    for (const string& word : suggestions) {
        cout << "  " << word << "\n";
    }
    // Expected:
    //   car
    //   care
    //   cart
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 10. LeetCode Practice Problems

Use these problems to reinforce rotations and string retrieval structures.

### AVL & Tree Balance

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 110 | [Balanced Binary Tree](https://leetcode.com/problems/balanced-binary-tree/) | Easy | Check height balance invariant at every node recursively |
| 108 | [Convert Sorted Array to Binary Search Tree](https://leetcode.com/problems/convert-sorted-array-to-binary-search-tree/) | Easy | Construct balanced BST by picking the middle element recursively as root |
| 1382 | [Balance a Binary Search Tree](https://leetcode.com/problems/balance-a-binary-search-tree/) | Medium | Extract nodes inorder to get sorted vector, then reconstruct balanced tree using #108 |

### Trie Implementations

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 208 | [Implement Trie (Prefix Tree)](https://leetcode.com/problems/implement-trie-prefix-tree/) | Medium | Section 6, directly — implement standard insert, search, and startsWith |
| 211 | [Design Add and Search Words Data Structure](https://leetcode.com/problems/design-add-and-search-words-data-structure/) | Medium | Trie search with wildcard support. If character is `.`, recurse on all non-null children |
| 677 | [Map Sum Pairs](https://leetcode.com/problems/map-sum-pairs/) | Medium | Store values in TrieNodes. For prefix queries, traverse to prefix node and sum child nodes |
| 212 | [Word Search II](https://leetcode.com/problems/word-search-ii/) | Hard | Advanced combination of Grid DFS Backtracking (`03_Backtracking` Section 6) and Trie matching — store target dictionary in Trie to discard prefix search directions |

### Self-Check Before Moving to `05_Segment_and_Fenwick_Trees`
If you can calculate node balance factors, trace single/double rotations, write Trie insertion/search, and explain why Trie lookup is $O(L)$, you have completed the goals of this notebook. In `05_Segment_and_Fenwick_Trees`, we will explore advanced hierarchical array structures designed to optimize range queries and point updates.